# Test

In [1]:
import numpy as np
from enum import IntFlag
import math
import sounddevice as sd

In [10]:

def message_to_bits(msg:str):
    bits = []
    for char in msg:
        bits.extend([int(bit) for bit in format(ord(char), '08b')])
    return bits

class Verbosity(IntFlag):
    QUITE = 0
    INFO = 1 << 0
    WARNING = 1 << 1
    ERROR = 1 << 2
    ALL = 0xFFFFFF

class Note:
    def __init__(self, f):
        # The frequency used by the note
        self.frequency = f

    def generate_signal(self, t, amplitude:float=1):
        omega = 2 * np.pi * self.frequency
        return amplitude * np.sin(t * omega)

class NoteUplet:
    def __init__(self, cardinality, r):

        # The number of elements in the n-uplet
        self.cardinality = cardinality

        # An array contaning the notes
        self.notes = None
        self.populateNotes(r)
    
    def populateNotes(self, r=(0, 1)):
       self.notes = [Note(f) for f in np.linspace(r[0], r[1], self.cardinality)]

class MessageSubmitInfo:
    def __init__(self, cardinality=8):

        # How many infos are printed out
        self.verbosity = Verbosity.ALL

        # The sampling frequency of the machine
        self.sampling_frequency = 44100

        # The number of n-uplet sent per second
        self.transmit_frequency = 2
        
        # The used frequency range
        self.range = (300, 500)

        # The number of simultanious bits, by default 8 but can be extented if the frequency range allows it
        self.notes = NoteUplet(cardinality, self.range)
    
    def debug_print(self):
        print("==== Message submit info ====")
        print("\tsampling frequency : ", self.sampling_frequency)
        print("\ttransmit frequency : ", self.transmit_frequency)
        print("\tverbosity : ", self.verbosity)
        print("\trange : ", self.range)
    
    def encodeMessage(self, message:str):
        lenght = len(message)

        # The time spent per signal
        period = 1/self.transmit_frequency

        # The number of samples per notes
        note_samples = self.sampling_frequency * period

        # Convert message to an array of bits
        bits = message_to_bits(message)
        
        # The output signal
        output = np.zeros(0)
        
        t = np.linspace(0, period, int(note_samples))
        signal = np.zeros(int(note_samples))

        for i in range(0, len(bits)):
            bit = bits[i]

            index = i % self.notes.cardinality
            print(index)
            if index == 0 and i != 0:
                output = np.append(output, signal)
                signal = np.zeros(int(note_samples))

            if bit == 0: continue

            
            signal += self.notes.notes[index].generate_signal(t)

        output = np.append(output, signal)
        return output

    def stream_message(self, msg:str):
        lenght = len(msg)

        # The time spent per signal
        period = 1/self.transmit_frequency

        # The number of samples per notes
        note_samples = self.sampling_frequency * period

        signal = np.zeros(int(note_samples))

        cardinality = self.notes.cardinality

        index = 0
        t = np.linspace(0, period, int(note_samples))

        for i in range(len(msg)*8):


            char = msg[int(i/8)]

            b = (ord(char) >> (7 - (i % 8))) & 1

            index = (i+1) % cardinality
            if i%8 == 0:
                print(char, [int(bit) for bit in format(ord(char), '08b')])

            print(b, end='')
            if b == 1:
                signal += self.notes.notes[index].generate_signal(t)

            if index == 0:
                print(char)
                sd.wait()

                sd.play(signal.copy(), samplerate=self.sampling_frequency, blocking=False)
                # sd.play(signal.copy(), samplerate=self.sampling_frequency)
                signal = np.zeros(int(note_samples))
            
        # sd.play(signal.copy(), samplerate=self.sampling_frequency)
        sd.wait()
    
    def stream_test(self, message:str):
        lenght = len(message)

        # The time spent per signal
        period = 1/self.transmit_frequency

        # The number of samples per notes
        note_samples = self.sampling_frequency * period

        signal = np.zeros(int(note_samples))

        cardinal = self.notes.cardinality

        t = np.linspace(0, period, int(note_samples))

        
        def callback(outdata, frames, time, status):
            if status:
                print(status)  # If there's an error, print the status
            
            signal = np.zeros(int(frames))  # Initialize the output signal


            # Calculate the start and end points based on current time and frame count
            start = int(time.outputBufferDacTime / note_samples)  # Determine the starting index in the message
            end = start + int(frames // note_samples)  # Calculate the end index for the current segment

            # Iterate through the message based on calculated start and end positions
            for i in range(start, end+1):
                char = message[i % len(message)]  # Get the character for the current bit
                b = (ord(char) >> (7 - (i % 8))) & 1  # Extract the bit (0 or 1)

                # Calculate the note index by cycling through the notes
                index = i % cardinal

                # Print binary representation of the character when a new character is encountered
                if i % 8 == 0:
                    print(char, [int(bit) for bit in format(ord(char), '08b')])  # Print binary representation of the character

                print(b, end='')  # Print the bit value for debugging

                # If the bit is 1, generate the signal for the current note and accumulate it
                if b == 1:
                    signal += self.notes.notes[index].generate_signal(t)

                # Every 8 bits, play the signal for this character and reset it
                if (i + 1) % 8 == 0:
                    print(f"\nPlayed signal for char: {char}")
                    # sd.play(signal.copy(), samplerate=self.sampling_frequency, blocking=False)  # Play the accumulated signal for the char
                    signal = np.zeros(int(note_samples))  # Reset the signal after playing

            # Fill the output buffer with the signal for the current frames
            outdata[:,0] = signal[:frames]
        
        with sd.OutputStream(callback=callback, channels=1, samplerate=self.sampling_frequency):
            # print("streaming audio")
            # print(period * lenght)
            sd.sleep(int(period * lenght))
                

submitInfo = MessageSubmitInfo(cardinality=8)
# submitInfo.stream_message("abcdefghijklmnopqrstuvwxyz")

sd.play(submitInfo.encodeMessage("CACA"*8))
sd.wait()

0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
0
1
2
3
4
5
6
7
